# Multi-Robot Task Negotiation Engine — ML Prototype

This notebook builds the first ML component for HackFusion 2026: a **robot-task suitability model**.

**Updated in this revision:**
- Fixed a data-leakage issue where the label was a deterministic function of the exact input features (the model was just re-deriving an if/else rule, not learning anything — see Step 3 for details).
- Added realistic noise to simulate sensor/estimation uncertainty.
- Added cross-validation, ROC-AUC, and a baseline model for comparison.
- Added an overfitting check (train vs. test gap).
- Made `display()` safe to run outside Jupyter/Colab.
- Model is now saved as a metadata bundle (features, metrics, version) for safer API integration.

Workflow:
1. Setup & imports
2. Generate synthetic robot-task data (with realistic noise)
3. Create the training target (probabilistic, not a hard leak-prone rule)
4. Explore and prepare the data
5. Train/test split
6. Train a baseline (Logistic Regression) and a Random Forest, with cross-validation
7. Evaluate both models (accuracy, F1, ROC-AUC, overfitting check)
8. Feature importance
9. Predict suitability for new robots/tasks
10. Rank multiple robots for a task
11. Save the trained model bundle for later API/website integration


## Step 1 — Setup & imports

In [ ]:
# Uncomment the line below if running in a fresh environment (Colab, etc.)
# %pip install -q pandas numpy scikit-learn matplotlib joblib

import warnings
warnings.filterwarnings('ignore')

import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib

# display() only exists by default inside Jupyter/IPython. This makes it work
# as a plain script too, so the notebook can't break outside Colab/Jupyter.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

print('Environment ready! scikit-learn / pandas / numpy imported, seed set to', RANDOM_STATE)

## Step 2 — Generate synthetic robot-task data

Same feature set as before, but we now also keep a **noisy, observed** version of the features the model actually sees, separate from the clean 'ground truth' values used to decide the label. This mirrors reality: sensors, battery estimates, and workload readings are never perfectly precise.

In [ ]:
n = 10000

# 'True' underlying values (used to decide the ground-truth outcome)
true_distance = np.random.uniform(10, 1000, n)
true_battery = np.random.uniform(10, 100, n)
true_speed = np.random.uniform(1, 5, n)
true_workload = np.random.uniform(0, 100, n)
task_priority = np.random.randint(1, 11, n)
true_task_distance = np.random.uniform(50, 1500, n)
true_load_capacity = np.random.uniform(5, 100, n)
required_capacity = np.random.uniform(5, 100, n)

# Observed values = true values + realistic sensor/estimation noise.
# This is what the model is actually trained and evaluated on.
def add_noise(arr, noise_pct, low=None, high=None):
    noisy = arr + np.random.normal(0, noise_pct * np.abs(arr).mean(), size=arr.shape)
    if low is not None or high is not None:
        noisy = np.clip(noisy, low, high)
    return noisy

data = pd.DataFrame({
    'distance': add_noise(true_distance, 0.05, 1, None),
    'battery': add_noise(true_battery, 0.03, 0, 100),
    'speed': add_noise(true_speed, 0.05, 0.1, None),
    'workload': add_noise(true_workload, 0.05, 0, 100),
    'task_priority': task_priority,
    'task_distance': add_noise(true_task_distance, 0.05, 1, None),
    'load_capacity': add_noise(true_load_capacity, 0.03, 0, None),
    'required_capacity': required_capacity,
})

data['estimated_time'] = data['distance'] / data['speed']
data['energy_required'] = data['task_distance'] * 0.02 + data['required_capacity'] * 0.1

data.head()

## Step 3 — Create the training target

**Why this changed:** the original rule computed `suitable` directly from the same columns the model was trained on (e.g. `battery > energy_required`). That means the 'ML model' was mathematically guaranteed to reconstruct that if/else statement almost perfectly — the 98%+ accuracy you'd see isn't evidence of learning, it's evidence of leakage. A judge who tests edge cases exactly on the rule boundary would immediately see the model is just the rule in disguise.

**Fix:** we compute the label from the **clean/true** values (not the noisy observed ones the model sees), and turn the hard thresholds into a **soft, probabilistic** decision via a logistic (sigmoid) function, then sample the actual outcome from that probability. This creates realistic irreducible uncertainty near the decision boundary — the model has to learn a genuine pattern instead of memorizing a formula, and near-boundary cases will legitimately be uncertain, which is exactly what you'd expect from real robot/task data.

In [ ]:
true_energy_required = true_task_distance * 0.02 + required_capacity * 0.1
true_estimated_time = true_distance / true_speed

# Normalize each factor onto a roughly comparable scale, then combine into a weighted score.
battery_margin = (true_battery - true_energy_required) / 100.0
workload_ok = (80 - true_workload) / 100.0
capacity_margin = (true_load_capacity - required_capacity) / 100.0
time_ok = (500 - true_estimated_time) / 500.0

score = (
    1.6 * battery_margin +
    1.2 * workload_ok +
    1.4 * capacity_margin +
    1.0 * time_ok
)

# Standardize the score before squashing. Without this, the raw weighted sum skews
# heavily positive and produces a ~80/20 class split, which makes the model look
# accurate while actually just predicting the majority class most of the time.
# Centering keeps the same *direction* of the rule but gives a realistic, closer-to-
# balanced split so both classes are actually learnable.
score_standardized = (score - score.mean()) / score.std()
probability_suitable = 1 / (1 + np.exp(-1.4 * score_standardized))
data['suitable'] = (np.random.uniform(size=n) < probability_suitable).astype(int)

print('Dataset shape:', data.shape)
print('\nClass distribution:')
print(data['suitable'].value_counts())
print(f"\nLabel is now sampled from a probability (mean p={probability_suitable.mean():.3f}), "
      "not a hard rule on the model's own input columns.")

data.head()

## Step 4 — Inspect the data

In [ ]:
print(data.info())
print('\nMissing values:')
print(data.isnull().sum())

data.describe()

## Step 5 — Visualize the target distribution and feature correlations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

data['suitable'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Robot-Task Suitability Distribution')
axes[0].set_xlabel('Suitable (0 = No, 1 = Yes)')
axes[0].set_ylabel('Number of samples')
axes[0].tick_params(axis='x', rotation=0)

corr = data.corr(numeric_only=True)
im = axes[1].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr.columns)))
axes[1].set_xticklabels(corr.columns, rotation=90)
axes[1].set_yticks(range(len(corr.columns)))
axes[1].set_yticklabels(corr.columns)
axes[1].set_title('Feature Correlation Matrix')
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Step 6 — Prepare features and target

In [ ]:
features = [
    'distance',
    'battery',
    'speed',
    'workload',
    'task_priority',
    'task_distance',
    'load_capacity',
    'required_capacity',
    'estimated_time',
    'energy_required'
]

X = data[features]
y = data['suitable']

print('Features:', features)
print('X shape:', X.shape)
print('y shape:', y.shape)

## Step 7 — Split into training and testing data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## Step 8 — Train a baseline model and the Random Forest, with cross-validation

We train a simple Logistic Regression **baseline** alongside the Random Forest. This matters for your presentation: if the Random Forest doesn't meaningfully beat a simple linear baseline, that's important to know (and honestly report) rather than assume the fancier model is automatically better. We also use 5-fold stratified cross-validation instead of trusting a single train/test split, since a single split can get lucky or unlucky.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline_cv_scores = cross_val_score(baseline, X_train, y_train, cv=cv, scoring='f1')

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,           # capped depth: prevents the trees from just memorizing noise
    min_samples_leaf=15,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')

print(f'Baseline (Logistic Regression) CV F1: {baseline_cv_scores.mean():.4f} ± {baseline_cv_scores.std():.4f}')
print(f'Random Forest CV F1:            {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}')

# Fit both on the full training set for downstream evaluation/use
baseline.fit(X_train, y_train)
model.fit(X_train, y_train)
print('\nBoth models trained on the full training set.')

## Step 9 — Evaluate the model

Beyond accuracy, we check ROC-AUC (threshold-independent quality) and compare train vs. test performance to catch overfitting — a small gap is expected and healthy; a large one means the model memorized the training set.

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_proba)

train_accuracy = accuracy_score(y_train, model.predict(X_train))

print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test F1 Score: {test_f1:.4f}')
print(f'Test ROC-AUC:  {test_auc:.4f}')
print(f'\nTrain accuracy: {train_accuracy:.4f}  |  Test accuracy: {test_accuracy:.4f}  '
      f'|  Gap: {train_accuracy - test_accuracy:.4f}')
if train_accuracy - test_accuracy > 0.08:
    print('Warning: noticeable train/test gap - consider more regularization (lower max_depth, higher min_samples_leaf).')
else:
    print('Train/test gap looks healthy - model is not obviously overfitting.')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('Confusion Matrix (Random Forest)')
plt.show()

## Step 10 — Feature importance

In [ ]:
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance)

importance.plot(x='feature', y='importance', kind='bar', legend=False)
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 11 — Predict one new robot-task pair

In [ ]:
new_robot = pd.DataFrame({
    'distance': [150],
    'battery': [85],
    'speed': [3],
    'workload': [20],
    'task_priority': [8],
    'task_distance': [400],
    'load_capacity': [80],
    'required_capacity': [40],
})
new_robot['estimated_time'] = new_robot['distance'] / new_robot['speed']
new_robot['energy_required'] = new_robot['task_distance'] * 0.02 + new_robot['required_capacity'] * 0.1
new_robot = new_robot[features]  # enforce correct column order

prediction = model.predict(new_robot)[0]
probability = model.predict_proba(new_robot)[0]

print('Prediction:', 'Suitable' if prediction == 1 else 'Not Suitable')
print(f'Not Suitable probability: {probability[0]:.2%}')
print(f'Suitable probability: {probability[1]:.2%}')

## Step 12 — Rank multiple robots for one task

In [ ]:
robots = pd.DataFrame({
    'robot_id': ['R01', 'R02', 'R03', 'R04', 'R05'],
    'distance': [120, 300, 80, 500, 220],
    'battery': [90, 75, 65, 95, 55],
    'speed': [3.5, 2.5, 4.0, 3.0, 2.0],
    'workload': [20, 50, 30, 10, 60],
    'task_priority': [8, 8, 8, 8, 8],
    'task_distance': [500, 500, 500, 500, 500],
    'load_capacity': [80, 100, 60, 90, 70],
    'required_capacity': [40, 40, 40, 40, 40]
})

robots['estimated_time'] = robots['distance'] / robots['speed']
robots['energy_required'] = robots['task_distance'] * 0.02 + robots['required_capacity'] * 0.1

robot_features = robots[features]
robots['suitable'] = model.predict(robot_features)
robots['suitability_score'] = model.predict_proba(robot_features)[:, 1]

ranking = robots.sort_values('suitability_score', ascending=False)

print('Robot ranking for the task:')
display(ranking[['robot_id', 'battery', 'workload', 'distance', 'suitable', 'suitability_score']])

## Step 13 — Save the trained model

Saved as a **bundle** (model + feature list + metrics + version/timestamp) rather than just the raw model object. When you wire this into a FastAPI endpoint later, the feature list guarantees you build the input vector in the exact order the model expects, and the metrics travel with the artifact so you always know what you deployed.

In [ ]:
model_bundle = {
    'model': model,
    'features': features,
    'metrics': {
        'test_accuracy': float(test_accuracy),
        'test_f1': float(test_f1),
        'test_roc_auc': float(test_auc),
        'cv_f1_mean': float(rf_cv_scores.mean()),
        'cv_f1_std': float(rf_cv_scores.std()),
    },
    'trained_at_utc': datetime.now(timezone.utc).isoformat(),
    'sklearn_python_version': platform.python_version(),
    'random_state': RANDOM_STATE,
}

model_path = MODEL_DIR / 'robot_task_model.pkl'
joblib.dump(model_bundle, model_path)

print(f'Model bundle saved to: {model_path.resolve()}')
print(json.dumps(model_bundle['metrics'], indent=2))

## Step 14 — Load the saved model and test it

In [ ]:
loaded_bundle = joblib.load(model_path)
loaded_model = loaded_bundle['model']
loaded_features = loaded_bundle['features']

assert loaded_features == features, 'Feature order mismatch between saved bundle and current session!'

test_prediction = loaded_model.predict(new_robot[loaded_features])[0]
print('Loaded model prediction:', 'Suitable' if test_prediction == 1 else 'Not Suitable')
print('Loaded model metrics:', loaded_bundle['metrics'])

## Next stage

This notebook creates the first ML component, now with leakage fixed and proper validation. The next notebook stage should add:

- Collision-risk prediction
- Battery/energy prediction
- Robot failure simulation
- Deadlock detection
- Peer-to-peer negotiation
- 500+ robot simulation
- FastAPI endpoint for connecting the model to the website (load `models/robot_task_model.pkl` with `joblib.load`, and build the input row using `bundle['features']` to guarantee correct column order)
